# C2.10 · Case study — the Supabase pattern: open until closed

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Both directions*

Builds on **[C2.9 · Case study — Moltbook: 770,000 agents behind one missing policy](https://spbreed.github.io/cyber-commons/lessons/C2.9.html)**.

| | |
|---|---|
| Tools used | Supabase, PostgREST, sqlfluff |

## What this lesson is

**What it covers.** Audit a four-statement scaffold, then run the one catalogue query that answers the critical half across every table at once.

**Why a security engineer needs it.** A failure that is invisible in testing, because nothing about the application's behaviour is wrong. One write-up puts it at 73% of generated applications carrying at least one issue. The control it builds is: a schema check in CI rather than an application test — and, better, a default that does not expose a table until something opts it in.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Supabase has never been breached. Applications built on it leak constantly, and always through the same two doors — a table with no row policy, and an admin key in a frontend bundle. One write-up puts it at 73% of generated applications carrying at least one issue.

> **At CyberTravels.** The class behind Moltbook, and the one a generated scaffold at CyberTravels lands in by default: a table with no row policy, and an admin key in a frontend bundle.

## 2 · The framework

```
   two critical patterns, and they are not degrees of one thing

   RLS disabled on a table        service_role key in the frontend
   removes the POLICY             removes the POLICY ENGINE
   anon key reads that table      admin key reads every table

   why it recurs
   generator -> create table ... (no policy attached)
             -> frontend works perfectly
             -> every test passes
             -> nothing in the suite is shaped like the question

   the two fixes, and only one survives the next deadline
   per table   alter table ... enable row level security
   by default  not exposed through the Data API unless opted in   <- 2026
```

Moltbook is the instance. This is the class, and the class is bigger than any
one platform.

Supabase has not been breached. What has happened repeatedly is that
*applications built on it* expose their data, and the write-ups converge on a
small number of patterns — two of them critical:

**Tables without Row-Level Security.** RLS was historically opt-in. A table
created without it is, in the write-ups' own words, "completely open via the
public API": anyone with the anon key — which ships to every browser — can read
it, and depending on policy, write it.

**The `service_role` key in client code.** That key exists to bypass RLS for
trusted server-side work. In a frontend bundle it is a complete RLS bypass and
full administrative database access, regardless of how good the policies are.

Three more sit just below: overly permissive policies, RPC functions with no
authorisation check of their own, and unsecured storage buckets.

What makes this a lesson rather than a checklist is **why it recurs now**.
Code generators scaffold a project, create tables and write the frontend — and
happily emit `create table` with no policy attached, because nothing in the
prompt asked for one and the code works without it. The failure is invisible in
testing: the app functions perfectly. One write-up puts it at **73% of
"vibe-coded" applications carrying at least one security issue**, with secrets
the most common category.

The structural fix is the one worth taking away. From 2026, new Supabase
projects **no longer expose public-schema tables through the Data API by
default** — closing the gap at the level of the default rather than at the level
of everyone remembering. That is the same argument as default-deny on the tool
call (A3.1), arriving at a database.

> **Sources.** Public write-ups of the Supabase misconfiguration patterns and
> CVE-2025-48757: [VibeAppScanner](https://vibeappscanner.com/issues/supabase),
> [GuardLayer](https://www.guardlayer.io/blog/supabase-security-breaches).
> Supabase itself has not been breached; the pattern is in applications built
> on it, which is what makes it a lesson about defaults rather than about a
> vendor.

## 3 · The two critical patterns, and what each defeats

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">pattern</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what an attacker holds</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what it defeats</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">severity</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">table with RLS disabled</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the anon key, from any browser</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">all row-level access control on that table</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>critical</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><code>service_role</code> key in the frontend</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">an admin key</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">every policy, on every table</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>critical</b></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">overly permissive policy</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the anon key</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the intent of the policy, not its existence</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">high</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">RPC function with no auth check</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the anon key</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the policies the function bypasses</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">high</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">unsecured storage bucket</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the bucket URL</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">object-level access</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">high</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">The first two are not degrees of the same problem. One removes the policy; the other removes the policy engine.</div>

## 4 · Why generated code lands here by default

This is the part worth running: the vulnerable version and the safe version are indistinguishable from the application's behaviour, which is exactly why testing does not catch it.

In [ ]:
SCAFFOLD = [
 "create table profiles (id uuid primary key, email text, api_key text);",
 "create table posts    (id uuid primary key, author uuid, body text);",
 "alter table posts enable row level security;",
 "create policy p_read on posts for select using (true);",
]

def audit(statements):
    """The check a generator does not run, expressed as three questions."""
    tables = [s.split()[2] for s in statements if s.startswith("create table")]
    rls_on = {s.split()[2] for s in statements if "enable row level security" in s}
    permissive = [s.split()[2] for s in statements
                  if s.startswith("create policy") and "using (true)" in s]
    findings = []
    for tbl in tables:
        if tbl not in rls_on:
            findings.append((tbl, "critical", "RLS disabled - open via the public API"))
    for s in statements:
        if s.startswith("create policy") and "using (true)" in s:
            findings.append((s.split()[4], "high", "policy matches every row"))
    return tables, findings

tables, findings = audit(SCAFFOLD)
print(f"tables created : {', '.join(tables)}")
print(f"findings       : {len(findings)}\n")
print(f"{'table':12s}{'severity':11s}why")
for tbl, sev, why in findings:
    print(f"{tbl:12s}{sev:11s}{why}")

print()
print("The application works. Every feature passes. `profiles` holds an api_key")
print("column and has no policy at all, and nothing in the test suite is shaped")
print("like the question that would find it.")
assert any(f[1] == "critical" for f in findings)
assert "profiles" in [f[0] for f in findings]

## 5 · The detection that actually scales

In [ ]:
# One query answers the critical half, across every table at once.
CATALOG = [                       # what pg_class would return
 {"relname": "profiles", "relrowsecurity": False},
 {"relname": "posts",    "relrowsecurity": True},
 {"relname": "sessions", "relrowsecurity": False},
 {"relname": "audit",    "relrowsecurity": True},
]
SENSITIVE = {"profiles", "sessions", "audit"}

open_tables = sorted(r["relname"] for r in CATALOG if not r["relrowsecurity"])
print("select relname from pg_class where relrowsecurity = false;")
for name in open_tables:
    mark = "  <- holds credentials or session state" if name in SENSITIVE else ""
    print(f"   {name}{mark}")

print(f"\ntables open via the public API : {len(open_tables)} of {len(CATALOG)}")
exposed_sensitive = [t for t in open_tables if t in SENSITIVE]
print(f"of those, sensitive             : {len(exposed_sensitive)} "
      f"({', '.join(exposed_sensitive)})")
print()
print("This is a one-line query and it is the highest-value one in the whole")
print("pattern. It is also not something an application test can express, which")
print("is why it belongs in CI against the schema rather than in the test suite.")
assert exposed_sensitive == ["profiles", "sessions"]

## 6 · The fix that does not rely on anyone remembering

Two ways to close this. Only one of them survives the next engineer, the next generated scaffold, and the next deadline.

<svg viewBox="0 0 700 152" width="100%" style="max-width:700px;height:auto;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px"><defs><marker id="a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#8A93A6"/></marker></defs><rect x="6" y="14" width="320" height="96" rx="5" fill="none" stroke="#E0912F" stroke-width="1.4"/><text x="166.0" y="66.0" text-anchor="middle" fill="#E0912F">fix each table</text><text x="166" y="58" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">alter table … enable row level security</text><text x="166" y="76" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">create policy … using (auth.uid() = owner)</text><text x="166" y="96" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">correct, and true only until the next table</text><rect x="374" y="14" width="320" height="96" rx="5" fill="none" stroke="#3FA06B" stroke-width="1.4"/><text x="534.0" y="66.0" text-anchor="middle" fill="#3FA06B">fix the default</text><text x="534" y="58" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">tables are not exposed through the Data API</text><text x="534" y="76" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">unless something opts them in</text><text x="534" y="96" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">the Supabase default since 2026</text><text x="350" y="138" text-anchor="middle" fill="#8A93A6" font-size="11.5" font-weight="normal">the same argument as default-deny on the tool call (A3.1), arriving at a database</text></svg><div style="font-size:12px;color:#8A93A6;margin-top:2px">A control that depends on everyone remembering is a control with a half-life.</div>

## 7 · What this hands to the rest of the commons

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">finding here</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the control, and where it lives</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">credential-shaped data in a client-readable table</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A3.8 — shared infrastructure between agent runs</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">an admin key reachable from the client</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A3.8 — admin plane off the workload path</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">a default that is open until closed</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A3.1 — default-deny, applied to data rather than tools</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">a schema check no application test expresses</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">D1.9 — detections whose subject is the platform</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">73% of generated apps carrying at least one issue</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">B2.12 — securing the developers&#x27; coding agents</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Every row is a control that already exists in this curriculum. The case study's job was to show you why it is there.</div>

## What you just proved

An audit of a four-statement scaffold finds a critical issue: the `profiles` table holds an api_key column and has no RLS at all, while every feature of the application works. The catalogue query then finds two of four tables open via the public API, both of them holding credentials or session state — a one-line check that no application test expresses.

## Your turn

Take the last thing a code generator scaffolded for you and run the two questions from this lesson against it: which tables have no row policy, and is any admin-scoped key reachable from the client bundle? Neither question is about the framework — both are about what the default was when nobody said otherwise.

## Where this leaves you

**What you can do now.** Research that reproduces — model effect separated from harness effect, benchmarks checked for a floor, a leaked key and a loose matcher — a handover that ends in a control with an eval case that fails on the old build, and three real incidents worked end to end: an agent swarm, a platform that shipped its database open, and the default that made the third one ordinary.

**What you still cannot do.** You can now produce a finding, prove it, and hand it over. You still cannot see it happen in production: nothing here tells you that the class you closed is being attempted right now, by whom, or how fast — and the register you just built assigned twelve controls to a function you have not read yet.

**Function D is the operational half — detecting an actor that acts a thousand times an hour, and stopping it. Next → D1.0, what AI for security operations means.**

---

**Next → [D1.0 · Start here — what AI for security operations means](https://spbreed.github.io/cyber-commons/lessons/D1.0.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*